In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

In [25]:
# 1. LOAD DATA
df = pd.read_csv("../data/fraud_raw.csv")
print("Initial shape:", df.shape)
df.head()

Initial shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [26]:
# 2. DROP UNNECESSARY COLUMNS
# These columns do not help detect fraud (random IDs + useless flag)
cols_to_drop = [
    "nameOrig", 
    "nameDest", 
    "isFlaggedFraud"
]

df.drop(columns=cols_to_drop, inplace=True)
print("Shape after dropping ID + flagged columns:", df.shape)

Shape after dropping ID + flagged columns: (6362620, 8)


Irrelevant Variables:

NameOrig and NameDest were removed because they are unique identifiers with extremely high cardinality and do not contain meaningful behavioral information for fraud prediction.

isFlaggedFraud was removed because it is almost always zero and does not correlate with actual fraud cases.

step was evaluated and kept/dropped depending on whether time‑based fraud patterns were observed in EDA.

newbalanceOrg and newbalanceDest were evaluated for redundancy and removed if highly correlated with other balance variables.

In [27]:
# 3. FIX INCORRECT / IMPOSSIBLE VALUES
# Negative balances are impossible → convert to NaN
balance_cols = [
    "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest"
]

for col in balance_cols:
    df[col] = df[col].apply(lambda x: np.nan if x < 0 else x)

# Replace NaN balances with 0 (common in this dataset)
df[balance_cols] = df[balance_cols].fillna(0)

In [28]:
# 4. REMOVE OUTLIERS 
# Remove extreme transaction amounts
amount_threshold = df["amount"].quantile(0.999)
df = df[df["amount"] <= amount_threshold]

print("Shape after removing extreme outliers:", df.shape)

Shape after removing extreme outliers: (6356257, 8)


In [29]:
# 5. FIX FORMATTING ISSUES
# Ensure numeric columns are numeric
numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

# Ensure categorical column is properly typed
df["type"] = df["type"].astype("category")

In [30]:
# 6. CHECK FOR NULLS AND HANDLE THEM
print("Null values per column:")
print(df.isnull().sum())

# If any nulls remain, fill with 0
df = df.fillna(0)

Null values per column:
step              0
type              0
amount            0
oldbalanceOrg     0
newbalanceOrig    0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
dtype: int64


In [31]:
# 7. SAVE CLEANED DATAFRAME
df.to_csv("fraud_clean.csv", index=False)
print("Cleaned dataset saved as fraud_clean.csv")

Cleaned dataset saved as fraud_clean.csv
